<a href="https://www.kaggle.com/code/shahirhabib/graphfusion-2?scriptVersionId=314870808" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc
)
import pandas as pd
import numpy as np
import matplotlib as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np
import torch
from datasets import load_dataset
# from transformers import (
#     RobertaTokenizer,
#     RobertaForSequenceClassification,
#     Trainer,
#     TrainingArguments,
# )
# from transformers import AutoTokenizer, AutoConfig, RobertaModel, RobertaForSequenceClassification
# import torch
# from transformers import TrainingArguments, Trainer
def do_nothing_for_five_minutes():
    # Message to show the program has started its wait
    print("Starting a 5-minute wait at", time.strftime("%H:%M:%S"))
    
    # Sleep for 300 seconds (5 minutes * 60 seconds/minute)
    try:
        time.sleep(3)
    except KeyboardInterrupt:
        # Allows you to stop the script with Ctrl+C
        print("\nWait interrupted by user.")
        return

    # Message to show the program has finished
    print("Wait finished at", time.strftime("%H:%M:%S"))
 
do_nothing_for_five_minutes()

In [ ]:
!gdown --id 1P3vMh_q1va3Kfz4h2oHScy1avUkOeEPb -O /kaggle/working/gcb_test_embeddings.npz

In [ ]:
!gdown 1UvwvxWw5B_aRxut00lxj2GBK4hX1F-He -O /kaggle/working/gcb_train_embeddings.npz

In [ ]:
!gdown 1oQMH6ERTp6fe6F5uw5m0Raa2_qrESFW0 -O /kaggle/working/trainedmodel.zip

In [ ]:
!gdown --id 1up8AC6F0rp3pcvnKISPzhSLVVRW8p1ui -O /kaggle/working/finaltest_features_all.parquet

In [ ]:
!gdown --id 1zrI_UdIpuQHQKVqE_5coFfOb84MwCkif -O /kaggle/working/sampletest_features_all.parquet

In [ ]:
!unzip /kaggle/working/trainedmodel.zip -d gcb_finetuned

In [ ]:
# =============================
# 1. Imports
# =============================
import torch
from datasets import load_dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# =============================
# 2. Load dataset
# =============================
dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")

# Column names (CHANGE HERE if needed)
TEXT_COL = "code"
LABEL_COL = "label"

# =============================
# 3. Subsample (5000 train / 1000 val)
# =============================
train_ds = dataset["train"].shuffle(seed=42).select(range(10000))
val_ds   = dataset["validation"].shuffle(seed=42).select(range(1000))
test_ds  = dataset["test"]
print(train_ds)
print(val_ds)
print(test_ds)

# =============================
# 4. Tokenizer & model
# =============================
tokenizer = RobertaTokenizer.from_pretrained(
    "microsoft/graphcodebert-base"
)

model = RobertaForSequenceClassification.from_pretrained(
    "microsoft/graphcodebert-base",
    num_labels=2
)

# =============================
# 5. Tokenization function
# =============================
def tokenize_fn(batch):
    return tokenizer(
        batch[TEXT_COL],
        padding="max_length",
        truncation=True,
        max_length=512,
    )
    tokens["labels"] = batch[LABEL_COL]
    return tokens

# =============================
# 6. Tokenize datasets (keep labels!)
# =============================
train_ds = train_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=[TEXT_COL],  # <-- DO NOT remove labels after mapping
)

val_ds = val_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=[TEXT_COL],
)

train_ds.set_format("torch")
val_ds.set_format("torch")

# =============================
# 7. Training arguments
# =============================
training_args = TrainingArguments(
    output_dir="./graphcodebert_run",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
    save_steps=100,
    save_total_limit=2,
    fp16=True,                   # mixed precision (T4 friendly)
    dataloader_num_workers=2,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

# =============================
# 8. Trainer
# =============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)

# =============================
# 9. Train
# =============================
trainer.train()

# =============================
# 10. Evaluate
# =============================
metrics = trainer.evaluate()
print(metrics)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np

# =============================
# 1. Get predictions on test dataset
# =============================

# We assume that your model is already in evaluation mode
dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")

# Column names (CHANGE HERE if needed)
TEXT_COL = "code"
LABEL_COL = "label"
test_ds  = dataset["test"]
# Tokenize the test dataset using the same tokenize_fn
test_ds = test_ds.map(
    tokenize_fn,  # The same tokenization function
    batched=True,
    remove_columns=[TEXT_COL],  # Remove the text column, but keep labels
)

# Set format to torch (similar to train_ds and val_ds)
test_ds.set_format("torch")

pred_output = trainer.predict(test_ds)

# Logits → predicted labels
y_pred = np.argmax(pred_output.predictions, axis=1)

# True labels
y_true = pred_output.label_ids


# =============================
# 2. Print the classification report
# =============================
print("Classification Report:")
print(classification_report(y_true, y_pred))


# =============================
# 3. Plot the confusion matrix
# =============================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(cm.shape[0]),
    yticklabels=range(cm.shape[0])
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Fine tuning on subset of train

In [ ]:
# =============================
# 1. Imports
# =============================
import torch
from datasets import load_dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from transformers import AutoTokenizer, AutoConfig, RobertaModel, RobertaForSequenceClassification
import torch
from transformers import TrainingArguments, Trainer
# =============================
# 2. Load dataset
# =============================
dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")

# Column names (CHANGE HERE if needed)
TEXT_COL = "code"
LABEL_COL = "label"

# =============================
# 3. Subsample (5000 train / 1000 val)
# =============================
train_ds = dataset["train"].shuffle(seed=42).select(range(10000))
val_ds   = dataset["validation"].shuffle(seed=42).select(range(1000))
test_ds  = dataset["test"]
print(train_ds)
print(val_ds)
print(test_ds)

# =============================
# 4. Load model & tokenizer
# =============================
BASE_MODEL = "microsoft/graphcodebert-base"
CHECKPOINT = "jiekeshi/GraphCodeBERT-Adversarial-Finetuned-Clone-Detection"

# 1) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# 2) Load config for a classification model
config = AutoConfig.from_pretrained(BASE_MODEL, num_labels=2)

# 3) Load base encoder with MLM weights
base_encoder = RobertaModel.from_pretrained(
    CHECKPOINT,
    config=config,
    ignore_mismatched_sizes=True,  # allows missing classification head
)

# 4) Build a classification model using that encoder
model = RobertaForSequenceClassification.from_pretrained(
    BASE_MODEL,
    config=config,
)

# 5) Replace the classification encoder weights with the adversarial encoder
model.roberta = base_encoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# =============================
# 5. Tokenization function
# =============================
def tokenize_batch(batch):
    # tokenize the source code
    tokens = tokenizer(
        batch["code"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    # preserve the label
    tokens["labels"] = batch["label"]
    return tokens
# =============================
# 6. Tokenize datasets (keep labels!)
# =============================
train_dataset = train_ds.map(tokenize_batch, batched=True)
eval_dataset = val_ds.map(tokenize_batch, batched=True)

train_dataset = train_dataset.remove_columns(["code"])
eval_dataset = eval_dataset.remove_columns(["code"])
train_dataset.set_format("torch")
eval_dataset.set_format("torch")


# =============================
# 7. Training arguments
# =============================
training_args = TrainingArguments(
    output_dir="./gcb_finetuned",
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=100,
    save_steps=100,
    fp16=True,
    save_total_limit=2,
    dataloader_num_workers=2,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

# =============================
# 8. Trainer
# =============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
     tokenizer=tokenizer,
)
# =============================
# 9. Train
# =============================
trainer.train()
# =============================
# 10. Evaluate
# =============================
metrics = trainer.evaluate()
print(metrics)

## sanity check

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np

# =============================
# 1. Get predictions on test dataset
# =============================

# We assume that your model is already in evaluation mode
dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")

# Column names (CHANGE HERE if needed)
TEXT_COL = "code"
LABEL_COL = "label"
test_ds  =  dataset["test"]
# Tokenize the test dataset using the same tokenize_fn
test_ds = test_ds.map(
    tokenize_batch,  # The same tokenization function
    batched=True,
    remove_columns=[TEXT_COL],  # Remove the text column, but keep labels
)
# Set format to torch (similar to train_ds and val_ds)
test_ds.set_format("torch")

pred_output = trainer.predict(test_ds)

# Logits → predicted labels
y_pred = np.argmax(pred_output.predictions, axis=1)

# True labels
y_true = pred_output.label_ids


# =============================
# 2. Print the classification report
# =============================
print("Classification Report:")
print(classification_report(y_true, y_pred))


# =============================
# 3. Plot the confusion matrix
# =============================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(cm.shape[0]),
    yticklabels=range(cm.shape[0])
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Obtaining Embeddings

In [ ]:
from transformers import AutoModel
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from transformers import AutoTokenizer, AutoConfig, RobertaModel, RobertaForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
finetuned_model = AutoModel.from_pretrained("/kaggle/working/gcb_finetuned/kaggle/working/gcb_finetuned/checkpoint-1875")
finetuned_model.to(device)
finetuned_model.eval()

tokenizer = RobertaTokenizer.from_pretrained(
    "microsoft/graphcodebert-base"
)
def get_embedding(code_snippet):
    inputs = tokenizer(
        code_snippet,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = finetuned_model(**inputs)

    # Mean pooling over the token dimension to get fixed size
    last_hidden_state = outputs.last_hidden_state
    embedding = last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding
embeddings = []
labels = []

code_str = """def get_embedding(code_snippet):
    inputs = tokenizer(
        code_snippet,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = finetuned_model(**inputs)

    # Mean pooling over the token dimension to get fixed size
    last_hidden_state = outputs.last_hidden_state
    embedding = last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding"""
emb_vec = get_embedding(code_str)
print(emb_vec.shape)
# for e in tqdm(train_ds) :
#     emb_vec = get_embedding(e['code'])
#     embeddings.append(emb_vec)
#     labels.append(e['label'])
    
# print(len(embeddings))
# print(labels)

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
finetuned_model.eval()
dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
test_ds = dataset["test"]
print(test_ds)

In [ ]:
import pandas as pd
from datasets import Dataset
finaltestdf = pd.read_parquet("/kaggle/working/finaltest_features_all.parquet")
finaltestdf.info()
finaltestdf.head()
tdf = finaltestdf
dataset = Dataset.from_pandas(finaltestdf[['code', 'ID', 'language']], preserve_index=False)

print(dataset)
print(dataset.features)

In [ ]:
tdf = finaltestdf
tdf.info()

In [ ]:
def collate_fn(batch):
    codes = [item["code"] for item in batch]
    labels = [item["ID"] for item in batch]  # using ID as label

    tokens = tokenizer(
        codes,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    return tokens, torch.tensor(labels, dtype=torch.long)
def mean_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size())
    masked = hidden_states * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1)
    return summed / counts
    
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=(device.type == "cuda")
)
embeddings = []
labels = []

use_amp = device.type == "cuda"

with torch.no_grad():
    for tokens, batch_labels in tqdm(loader):
        tokens = {k: v.to(device, non_blocking=True) for k, v in tokens.items()}
        batch_labels = batch_labels.to(device, non_blocking=True)

        if use_amp:
            with torch.cuda.amp.autocast(dtype=torch.float16):
                outputs = finetuned_model(**tokens)
        else:
            outputs = finetuned_model(**tokens)

        pooled = mean_pooling(
            outputs.last_hidden_state,
            tokens["attention_mask"]
        )

        embeddings.append(pooled.cpu())
        labels.append(batch_labels.cpu())
embeddings = torch.cat(embeddings).numpy()
labels = torch.cat(labels).numpy()

print("Final shape:", embeddings.shape)

In [ ]:
np.savez_compressed(
    "gcb_finaltest_embeddings.npz",
    embeddings=embeddings,
    ids=labels
)
data = np.load("gcb_finaltest_embeddings.npz")
X = data["embeddings"]
y = data["ids"]
print(X.shape,y.shape)

### Without id

In [ ]:
def mean_pooling(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(hidden_states.size())
    masked = hidden_states * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1)
    return summed / counts
    
def collate_fn(batch):
    codes = [item["code"] for item in batch]
    labels = [item["label"] for item in batch]
    tokens = tokenizer(
        list(codes),
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    return tokens, torch.tensor(labels, dtype=torch.long)
device = "cuda"
finetuned_model.eval()

loader = DataLoader(
    test_ds,
    batch_size=32,      # try 64 if memory allows
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)

embeddings = []
labels = []

with torch.no_grad():
    for tokens, batch_labels in tqdm(loader):
        tokens = {k: v.to(device, non_blocking=True) for k, v in tokens.items()}

        with torch.cuda.amp.autocast(dtype=torch.float16):
            outputs = finetuned_model(**tokens)
            pooled = mean_pooling(
                outputs.last_hidden_state,
                tokens["attention_mask"]
            )

        embeddings.append(pooled.cpu())
        labels.append(batch_labels)

embeddings = torch.cat(embeddings).numpy()
labels = torch.cat(labels).numpy()

print("Final shape:", embeddings.shape)

In [ ]:
np.savez_compressed(
    "gcb_test_embeddings.npz",
    embeddings=embeddings,
    labels=labels
)
data = np.load("gcb_test_embeddings.npz")
X = data["embeddings"]
y = data["labels"]

# Checking the discrimination

In [ ]:
# 1. Load the data
data = np.load("gcb_train_embeddings.npz")
X = data["embeddings"]
y = data["labels"]
import numpy as np
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

datat = np.load("gcb_test_embeddings.npz")     
X_test = datat["embeddings"]
y_test = datat["labels"]
X_test = scaler.transform(X_test)

datat = np.load("gcb_finaltest_embeddings.npz")     
Xf_test = datat["embeddings"]
yf_test = datat["ids"]
Xf_test = scaler.transform(Xf_test)

In [ ]:
# data = np.load("gcb_train_embeddings.npz")
# X, y = data["embeddings"], data["labels"]
print(X.shape, y.shape, np.unique(y, return_counts=True))


In [ ]:
from sklearn.metrics import silhouette_score

X,_, y,_ = train_test_split(X, y, test_size=0.95, random_state=42)
s = silhouette_score(X, y, metric="cosine")
# per-class centroids & inter/intra ratio
centroids = np.vstack([X[y==c].mean(0) for c in np.unique(y)])
intra = np.mean([np.linalg.norm(X[y==c]-centroids[i], axis=1).mean() for i,c in enumerate(np.unique(y))])
inter = np.mean([np.linalg.norm(centroids[i]-centroids[j]) for i in range(len(centroids)) for j in range(i+1,len(centroids))])
print(s, "intra:", intra, "inter:", inter, "ratio:", inter/intra)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
pipe = make_pipeline(StandardScaler(), PCA(n_components=9), LogisticRegression(max_iter=2000))
X_train,_, y_train,_ = train_test_split(X, y, test_size=0.9, random_state=42)
pipe.fit(X_train, y_train); print(pipe.score(X_test,y_test))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=5000,
        C=100,
        class_weight="balanced",
        solver="lbfgs"
    ))
])

clf.fit(X_train, y_train)
print("Train acc:", clf.score(X_train, y_train))
print("Test  acc:", clf.score(X_test, y_test))


In [ ]:
from sklearn.neighbors import NearestCentroid

nc = NearestCentroid(metric="euclidean")
nc.fit(X_train, y_train)
print("Train acc:", nc.score(X_train, y_train))
print("Test  acc:", nc.score(X_test, y_test))


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt


# load
data = np.load("gcb_train_embeddings.npz")
X_train, y_train = data["embeddings"], data["labels"]
# also load test
data_test = np.load("gcb_test_embeddings.npz")
X_test, y_test = data_test["embeddings"], data_test["labels"]
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
# 1) class counts
print("train counts:", np.unique(y_train, return_counts=True))
print("test  counts:", np.unique(y_test, return_counts=True))

# 2) PCA overlay (visual)
pca = PCA(n_components=10, random_state=0).fit(X_train)
Xp_tr = pca.transform(X_train)[:,:2]
Xp_te = pca.transform(X_test)[:,:2]
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.scatter(Xp_tr[:,0], Xp_tr[:,1], c=y_train, s=4, alpha=0.6); plt.title("PCA2 Train")
plt.subplot(1,2,2)
plt.scatter(Xp_te[:,0], Xp_te[:,1], c=y_test, s=4, alpha=0.6); plt.title("PCA2 Test")
plt.show()

# 3) KS tests on first 10 PCA components (train vs test overall)
for i in range(10):
    stat, p = ks_2samp(pca.transform(X_train)[:,i], pca.transform(X_test)[:,i])
    print(f"PC{i+1}: KS-stat={stat:.3f}, p={p:.3e}")

# 4) class-conditional centroid shift and normalized ratio
cent_train = {c: X_train[y_train==c].mean(0) for c in np.unique(y_train)}
cent_test  = {c: X_test[y_test==c].mean(0) for c in np.unique(y_test)}
for c in cent_train:
    inter = np.linalg.norm(cent_train[c] - cent_test[c])
    intra = np.mean(np.linalg.norm(X_train[y_train==c] - cent_train[c], axis=1))
    print(f"class {c} centroid shift {inter:.3f}, intra {intra:.3f}, ratio {inter/intra:.3f}")

# 5) Nearest-neighbour agreement: for each test sample, nearest train label
nbrs = NearestNeighbors(n_neighbors=1, metric="cosine").fit(X_train)
dist, idx = nbrs.kneighbors(X_test, return_distance=True)
pred_by_nn = y_train[idx.flatten()]
nn_acc = (pred_by_nn == y_test).mean()
print("1-NN (cosine) agreement:", nn_acc)

# 6) MMD quick (RBF) two-sample statistic (simple)
def rbf_mmd(x, y, gamma=1.0):
    xx = np.exp(-gamma * pairwise_distances(x, x, metric='sqeuclidean'))
    yy = np.exp(-gamma * pairwise_distances(y, y, metric='sqeuclidean'))
    xy = np.exp(-gamma * pairwise_distances(x, y, metric='sqeuclidean'))
    return xx.mean() + yy.mean() - 2*xy.mean()
print("MMD (train vs test):", rbf_mmd(X_train[:2000], X_test[:2000], gamma=1.0/X_train.shape[1]))


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
knn.fit(X_train, y_train)
print("KNN test acc:", knn.score(X_test, y_test))

In [ ]:
def coral(Xs, Xt):
    Cs = np.cov(Xs, rowvar=False) + np.eye(Xs.shape[1])
    Ct = np.cov(Xt, rowvar=False) + np.eye(Xt.shape[1])

    from scipy.linalg import fractional_matrix_power
    Xs_whiten = Xs @ fractional_matrix_power(Cs, -0.5)
    Xs_coral = Xs_whiten @ fractional_matrix_power(Ct, 0.5)
    return Xs_coral

X_train_coral = coral(X_train, X_test)

# then train classifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
print ("starting LR")
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, C=10))
])

clf.fit(X_train_coral, y_train)
print("Test acc:", clf.score(X_test, y_test))


In [ ]:
from sklearn.manifold import TSNE
sampledf = pd.DataFrame({ 
    'class': y_test
})
# ---- t-SNE ----
tsne = TSNE(n_components=2, perplexity=40, learning_rate='auto')
coords = tsne.fit_transform(X_test)
sampledf["tsne_x"], sampledf["tsne_y"] = coords[:,0], coords[:,1]

plt.figure(figsize=(7,5))
sns.scatterplot(data=sampledf, x='tsne_x', y='tsne_y', hue='class', s=5)
plt.title("t-SNE Projection")
plt.show()

In [ ]:
FINAL_FEATURES = [  
"ast_max_depth",
"ast_node_count",
"comment_ratio",
    
"loc_blank",
"ast_node_entropy",
"ast_avg_branching_factor",
    "snake_case_ratio",
    "loc_comments",
    "avg_identifier_length",
    "control_flow_ratio",
    "hapax_legomena_count",
    "unique_identifier_ratio",
    "cyclomatic_complexity",
    "avg_line_length",
     "halstead_volume",
    # "cognitive_complexity"
]
print(len(FINAL_FEATURES))

In [ ]:
import pandas as pd

# Convert X_test to DataFrame if it's a NumPy array
X_test_df = pd.DataFrame(X_test)

# Now concatenate with the selected features from tdf
combined_df = pd.concat([X_test_df, tdf[FINAL_FEATURES]], axis=1)

combined_df.info()

In [ ]:
sampledf.columns = sampledf.columns.astype(str)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import umap

# Your existing data preparation code

sampledf = combined_df
sampledf["class"] = y_train
# Step 1: Replace inf/-inf with NaN, then drop or fill
X_raw = sampledf.replace([np.inf, -np.inf], np.nan)

# Count how many infs were present originally
inf_count = 0

# Step 2: Fill NaNs with 0 (or you could drop rows if preferred)
X_clean = X_raw.fillna(0)

# Step 3: Scale
X = X_clean

# Get class labels
y = sampledf['class'].values
class_names = sampledf['class'].unique()

# # ---- UMAP ----
print("Generating UMAP...")
u = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(X)
sampledf["umap_x"], sampledf["umap_y"] = u[:,0], u[:,1]

# ---- PCA ----
print("Performing PCA...")
pca = PCA()
X_pca = pca.fit_transform(X)

# Store PCA components
sampledf["pca_x"], sampledf["pca_y"] = X_pca[:, 0], X_pca[:, 1]

# ---- LDA ----
print("Performing LDA...")
# For binary classification, LDA gives 1 component (n_classes - 1)
lda = LinearDiscriminantAnalysis()
try:
    X_lda = lda.fit_transform(X, y)
    
    # For binary classification, we get 1 component
    if X_lda.shape[1] == 1:
        # Create a 2D visualization by adding small random noise on y-axis
        np.random.seed(42)
        sampledf["lda_x"] = X_lda[:, 0]
        sampledf["lda_y"] = np.random.normal(0, 0.01, size=len(X_lda))
    else:
        # For multiclass with more than 2 classes
        sampledf["lda_x"], sampledf["lda_y"] = X_lda[:, 0], X_lda[:, 1]
    
    lda_success = True
except Exception as e:
    print(f"LDA failed: {e}")
    print("This can happen if classes are perfectly separable or covariance issues.")
    lda_success = False

# Create figure - adjust based on LDA success
if lda_success:
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    # Hide the LDA plot if it failed
    axes[1, 1].set_visible(False)

# 1. UMAP Plot
sns.scatterplot(data=sampledf, x='umap_x', y='umap_y', hue='class', s=5, ax=axes[0, 0])
axes[0, 0].set_title("UMAP Projection")
axes[0, 0].set_xlabel("UMAP 1")
axes[0, 0].set_ylabel("UMAP 2")

# 2. PCA Plot (first two components)
sns.scatterplot(data=sampledf, x='pca_x', y='pca_y', hue='class', s=5, ax=axes[0, 1])
axes[0, 1].set_title(f"PCA Projection\nPC1 ({pca.explained_variance_ratio_[0]*100:.1f}%) vs PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0, 1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0, 1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")

# 3. Scree Plot (Variance Explained)
components = range(1, len(pca.explained_variance_ratio_) + 1)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

axes[1, 0].bar(components[:15], pca.explained_variance_ratio_[:15], alpha=0.6, color='skyblue', label='Individual')
axes[1, 0].plot(components[:15], cumulative_variance[:15], 'ro-', label='Cumulative')
axes[1, 0].set_xlabel('Principal Component')
axes[1, 0].set_ylabel('Explained Variance Ratio')
axes[1, 0].set_title('Scree Plot (First 15 PCs)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Add variance percentages as text on bars for first few PCs
for i, (v, cv) in enumerate(zip(pca.explained_variance_ratio_[:5], cumulative_variance[:5])):
    axes[1, 0].text(i+1, v + 0.01, f'{v*100:.1f}%', ha='center', fontsize=9)
    if i < 4:
        axes[1, 0].text(i+1.3, cv - 0.02, f'{cv*100:.1f}%', ha='left', fontsize=9, color='red')

# 4. LDA Plot
if lda_success:
    if X_lda.shape[1] == 1:
        # 1D LDA - create jitter plot
        ax = axes[1, 1]
        for class_val in class_names:
            class_data = sampledf[sampledf['class'] == class_val]
            ax.scatter(class_data['lda_x'], class_data['lda_y'], 
                      s=10, alpha=0.6, label=class_val)
        ax.set_xlabel('LDA Component 1')
        ax.set_ylabel('Jitter (for visualization)')
        ax.set_title(f'LDA Projection (Binary Classification)\nLDA Accuracy: {lda.score(X, y)*100:.2f}%')
        ax.legend()
        ax.set_ylim(-0.05, 0.05)
        ax.yaxis.set_ticklabels([])  # Hide y-axis labels for jitter
    else:
        # 2D LDA for multiclass
        sns.scatterplot(data=sampledf, x='lda_x', y='lda_y', hue='class', s=5, ax=axes[1, 1])
        axes[1, 1].set_title(f'LDA Projection\nLDA Accuracy: {lda.score(X, y)*100:.2f}%')
        axes[1, 1].set_xlabel('LDA Component 1')
        axes[1, 1].set_ylabel('LDA Component 2')
    
    # Add decision boundary visualization for 1D LDA
    if X_lda.shape[1] == 1:
        # Find the decision boundary (where discriminant function = 0)
        # For binary LDA, the decision boundary is where the posterior probabilities are equal
        x_min, x_max = sampledf['lda_x'].min(), sampledf['lda_x'].max()
        x_range = np.linspace(x_min, x_max, 100).reshape(-1, 1)
        
        # Get predictions and probabilities
        preds = lda.predict(X)
        probs = lda.predict_proba(X)
        
        # Find the decision threshold
        # For LDA with equal priors, the decision boundary is where discriminant scores = 0
        # We can approximate it as the midpoint between class means
        mean_0 = sampledf[sampledf['class'] == class_names[0]]['lda_x'].mean()
        mean_1 = sampledf[sampledf['class'] == class_names[1]]['lda_x'].mean()
        boundary = (mean_0 + mean_1) / 2
        
        axes[1, 1].axvline(x=boundary, color='red', linestyle='--', alpha=0.7, 
                          label=f'Decision boundary\nx = {boundary:.3f}')
        axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"Number of infinity values omitted: {inf_count}")
print(f"Total variance explained by PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"Total variance explained by PC1+PC2: {sum(pca.explained_variance_ratio_[:2])*100:.2f}%")
print(f"Total variance explained by first 10 PCs: {sum(pca.explained_variance_ratio_[:10])*100:.2f}%")

if lda_success:
    print(f"\nLDA Results:")
    print(f"Training Accuracy: {lda.score(X, y)*100:.2f}%")
    print(f"Number of LDA components: {X_lda.shape[1]}")
    
    # For binary classification, show class separation along LDA axis
    if X_lda.shape[1] == 1:
        print("\nClass separation along LDA axis:")
        for class_val in class_names:
            class_scores = sampledf[sampledf['class'] == class_val]['lda_x']
            print(f"  {class_val}: Mean = {class_scores.mean():.3f}, Std = {class_scores.std():.3f}")
        
        # Calculate Fisher's discriminant ratio
        mean_0 = sampledf[sampledf['class'] == class_names[0]]['lda_x'].mean()
        mean_1 = sampledf[sampledf['class'] == class_names[1]]['lda_x'].mean()
        std_0 = sampledf[sampledf['class'] == class_names[0]]['lda_x'].std()
        std_1 = sampledf[sampledf['class'] == class_names[1]]['lda_x'].std()
         
        # Fisher's linear discriminant ratio
        fisher_ratio = (mean_0 - mean_1)**2 / (std_0**2 + std_1**2)
        print(f"\nFisher's discriminant ratio: {fisher_ratio:.3f}")
        print(f"Interpretation: Higher values indicate better separation")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import umap
# 2. Split into training and testing sets (80% train, 20% test)
X_train,_, y_train,_ = train_test_split(X, y, test_size=0.9, random_state=42)
X_train = X_test
y_train = y_test
sampledf = pd.DataFrame({
    'class': y_train
})
class_names = sampledf['class'].unique()

# ---- UMAP ----
print("Generating UMAP...")
u = umap.UMAP(n_neighbors=15, min_dist=0.1,n_components=3,metric="cosine",).fit_transform(X_train)
sampledf["umap_x"], sampledf["umap_y"] ,sampledf['umap_z'] = u[:,0], u[:,1] , u[:,2]


# ---- PCA ----
print("Performing PCA...")
pca = PCA()
X_pca = pca.fit_transform(X_train)

# Store PCA components
sampledf["pca_x"], sampledf["pca_y"] = X_pca[:, 0], X_pca[:, 1]

# ---- LDA ----
print("Performing LDA...")
# For binary classification, LDA gives 1 component (n_classes - 1)
lda = LinearDiscriminantAnalysis()
try:
    X_lda = lda.fit_transform(X_train, y_train)
    
    # For binary classification, we get 1 component
    if X_lda.shape[1] == 1:
        # Create a 2D visualization by adding small random noise on y-axis
        np.random.seed(42)
        sampledf["lda_x"] = X_lda[:, 0]
        sampledf["lda_y"] = np.random.normal(0, 0.01, size=len(X_lda))
    else:
        # For multiclass with more than 2 classes
        sampledf["lda_x"], sampledf["lda_y"] = X_lda[:, 0], X_lda[:, 1]
    
    lda_success = True
except Exception as e:
    print(f"LDA failed: {e}")
    print("This can happen if classes are perfectly separable or covariance issues.")
    lda_success = False

# Create figure - adjust based on LDA success
if lda_success:
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    # Hide the LDA plot if it failed
    axes[1, 1].set_visible(False)

sns.scatterplot(data=sampledf, x='umap_x', y='umap_y' , hue='class', s=5, ax=axes[0, 0])
axes[0, 0].set_title("UMAP Projection")
axes[0, 0].set_xlabel("UMAP 1")
axes[0, 0].set_ylabel("UMAP 2")

# 2. PCA Plot (first two components)
sns.scatterplot(data=sampledf, x='pca_x', y='pca_y', hue='class', s=5, ax=axes[0, 1])
axes[0, 1].set_title(f"PCA Projection\nPC1 ({pca.explained_variance_ratio_[0]*100:.1f}%) vs PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0, 1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0, 1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")

# 3. Scree Plot (Variance Explained)
components = range(1, len(pca.explained_variance_ratio_) + 1)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

axes[1, 0].bar(components[:15], pca.explained_variance_ratio_[:15], alpha=0.6, color='skyblue', label='Individual')
axes[1, 0].plot(components[:15], cumulative_variance[:15], 'ro-', label='Cumulative')
axes[1, 0].set_xlabel('Principal Component')
axes[1, 0].set_ylabel('Explained Variance Ratio')
axes[1, 0].set_title('Scree Plot (First 15 PCs)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Add variance percentages as text on bars for first few PCs
for i, (v, cv) in enumerate(zip(pca.explained_variance_ratio_[:5], cumulative_variance[:5])):
    axes[1, 0].text(i+1, v + 0.01, f'{v*100:.1f}%', ha='center', fontsize=9)
    if i < 4:
        axes[1, 0].text(i+1.3, cv - 0.02, f'{cv*100:.1f}%', ha='left', fontsize=9, color='red')

# 4. LDA Plot
if lda_success:
    if X_lda.shape[1] == 1:
        # 1D LDA - create jitter plot
        ax = axes[1, 1]
        colors = ['red', 'green']
        for i, class_val in enumerate( class_names):
            class_data = sampledf[sampledf['class'] == class_val]
            ax.scatter(class_data['lda_x'], class_data['lda_y'], 
                      s=10, alpha=0.6, label=class_val,color=colors[i % 2])
        ax.set_xlabel('LDA Component 1')
        ax.set_ylabel('Jitter (for visualization)')
        ax.set_title(f'LDA Projection (Binary Classification)\nLDA Accuracy: {lda.score(X_train, y_train)*100:.2f}%')
        ax.legend()
        ax.set_ylim(-0.05, 0.05)
        ax.yaxis.set_ticklabels([])  # Hide y-axis labels for jitter
    else:
        # 2D LDA for multiclass
        sns.scatterplot(data=sampledf, x='lda_x', y='lda_y', hue='class', s=5, ax=axes[1, 1])
        axes[1, 1].set_title(f'LDA Projection\nLDA Accuracy: {lda.score(X_train, y_train)*100:.2f}%')
        axes[1, 1].set_xlabel('LDA Component 1')
        axes[1, 1].set_ylabel('LDA Component 2')
    
    # Add decision boundary visualization for 1D LDA
    if X_lda.shape[1] == 1:
        # Find the decision boundary (where discriminant function = 0)
        # For binary LDA, the decision boundary is where the posterior probabilities are equal
        x_min, x_max = sampledf['lda_x'].min(), sampledf['lda_x'].max()
        x_range = np.linspace(x_min, x_max, 100).reshape(-1, 1)
        
        # Get predictions and probabilities
        preds = lda.predict(X_train)
        probs = lda.predict_proba(X_train)
        
        # Find the decision threshold
        # For LDA with equal priors, the decision boundary is where discriminant scores = 0
        # We can approximate it as the midpoint between class means
        mean_0 = sampledf[sampledf['class'] == class_names[0]]['lda_x'].mean()
        mean_1 = sampledf[sampledf['class'] == class_names[1]]['lda_x'].mean()
        boundary = (mean_0 + mean_1) / 2
        
        axes[1, 1].axvline(x=boundary, color='red', linestyle='--', alpha=0.7, 
                          label=f'Decision boundary\nx = {boundary:.3f}')
        axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"Total variance explained by PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"Total variance explained by PC1+PC2: {sum(pca.explained_variance_ratio_[:2])*100:.2f}%")
print(f"Total variance explained by first 10 PCs: {sum(pca.explained_variance_ratio_[:10])*100:.2f}%")

if lda_success:
    print(f"\nLDA Results:")
    print(f"Training Accuracy: {lda.score(X_train, y_train)*100:.2f}%")
    print(f"Number of LDA components: {X_lda.shape[1]}")
    
    # For binary classification, show class separation along LDA axis
    if X_lda.shape[1] == 1:
        print("\nClass separation along LDA axis:")
        for class_val in class_names:
            class_scores = sampledf[sampledf['class'] == class_val]['lda_x']
            print(f"  {class_val}: Mean = {class_scores.mean():.3f}, Std = {class_scores.std():.3f}")
        
        # Calculate Fisher's discriminant ratio
        mean_0 = sampledf[sampledf['class'] == class_names[0]]['lda_x'].mean()
        mean_1 = sampledf[sampledf['class'] == class_names[1]]['lda_x'].mean()
        std_0 = sampledf[sampledf['class'] == class_names[0]]['lda_x'].std()
        std_1 = sampledf[sampledf['class'] == class_names[1]]['lda_x'].std()
         
        # Fisher's linear discriminant ratio
        fisher_ratio = (mean_0 - mean_1)**2 / (std_0**2 + std_1**2)
        print(f"\nFisher's discriminant ratio: {fisher_ratio:.3f}")
        print(f"Interpretation: Higher values indicate better separation")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns

#Assuming X_test contains your embedding vectors and y_test contains true labels
#First, let's visualize with PCA
pca = PCA(n_components=4)
X_test_pca = pca.fit_transform(X_test)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='cool', alpha=0.6)
plt.title('Test Data - True Labels')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], alpha=0.6)
plt.title('Test Data - No Labels')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns

#Assuming X_test contains your embedding vectors and y_test contains true labels
#First, let's visualize with PCA
pca = PCA(n_components=4)
X_test_pca = pca.fit_transform(X_test)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='cool', alpha=0.6)
plt.title('Test Data - True Labels')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], alpha=0.6)
plt.title('Test Data - No Labels')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

plt.tight_layout()
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np

# If you haven't already: ensure PCA produced at least 3 components
# pca = PCA(n_components=4)
# X_test_pca = pca.fit_transform(X_test)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# Continuous or many classes: color by y_test
sc = ax.scatter(
    X_test_pca[:, 0], X_test_pca[:, 1], X_test_pca[:, 2],
    c=y_test, cmap='cool', alpha=0.7, s=40, edgecolor='k', linewidth=0.2
)

ax.set_title('3D PCA (PC1, PC2, PC3)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)')

plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.1, label='label value')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
X_train,_, y_train,_ = train_test_split(X, y, test_size=0.998, random_state=42)
colors = np.where(y_test == 0, "tab:blue", "tab:orange")
labels = ["Class 0", "Class 1"]
from sklearn.manifold import MDS

mds = MDS(
    n_components=2,
    metric=True,
    random_state=42,
    dissimilarity='euclidean'
)

X_mds = mds.fit_transform(X_test)

plt.figure(figsize=(6, 5))

for cls in [0, 1]:
    idx = y_test == cls
    plt.scatter(
        X_mds[idx, 0],
        X_mds[idx, 1],
        s=40,
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("MDS 1")
plt.ylabel("MDS 2")
plt.title("MDS (Colored by y_test)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial.distance import pdist, squareform

D = squareform(pdist(X_test, metric='euclidean'))
import numpy as np

def pcoa(distance_matrix, n_components=2):
    n = distance_matrix.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * H @ (distance_matrix ** 2) @ H
    eigenvalues, eigenvectors = np.linalg.eigh(B)

    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    L = np.diag(np.sqrt(np.maximum(eigenvalues[:n_components], 0)))
    V = eigenvectors[:, :n_components]

    return V @ L, eigenvalues

X_pcoa, eigvals = pcoa(D, n_components=2)

plt.figure(figsize=(6, 5))

for cls in [0, 1]:
    idx = y_test == cls
    plt.scatter(
        X_pcoa[idx, 0],
        X_pcoa[idx, 1],
        s=40,
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("PCoA 1")
plt.ylabel("PCoA 2")
plt.title("PCoA (Colored by y_test)")
plt.legend()
plt.tight_layout()
plt.show()
explained = eigvals / eigvals.sum()
print("PCoA variance explained:", explained[:2])


In [ ]:
from sklearn.decomposition import KernelPCA

kpca = KernelPCA(n_components=2, kernel='rbf', gamma=0.1)
X_kpca = kpca.fit_transform(X_test)

plt.figure(figsize=(6, 5))
for cls in [0, 1]:
    plt.scatter(
        X_kpca[y_test == cls, 0],
        X_kpca[y_test == cls, 1],
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("kPCA 1")
plt.ylabel("kPCA 2")
plt.title("Kernel PCA (RBF)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from sklearn.decomposition import KernelPCA

# Fit KPCA with 3 components
kpca = KernelPCA(n_components=3, kernel='rbf', gamma=0.1)
X_kpca = kpca.fit_transform(X_test)  # shape (n_samples, 3)

# Option A: color by y_test using a colormap (good for continuous labels or many classes)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(
    X_kpca[:, 0], X_kpca[:, 1], X_kpca[:, 2],
    c=y_test, cmap='cool', alpha=0.8, s=40, edgecolor='k', linewidth=0.2
)
ax.set_title('3D Kernel PCA (RBF) - colored by label')
ax.set_xlabel('kPCA 1')
ax.set_ylabel('kPCA 2')
ax.set_zlabel('kPCA 3')
plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.1, label='label value')
plt.tight_layout()
plt.show()
# Option B: discrete classes with separate colors and legend (good for small number of classes)
import numpy as np

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

unique_labels = np.unique(y_test)
colors = plt.cm.get_cmap('tab10', len(unique_labels))

for i, lab in enumerate(unique_labels):
    mask = (y_test == lab)
    ax.scatter(
        X_kpca[mask, 0], X_kpca[mask, 1], X_kpca[mask, 2],
        color=colors(i), label=str(lab), s=50, alpha=0.85, edgecolor='k', linewidth=0.2
    )

ax.set_title('3D Kernel PCA (RBF) by class')
ax.set_xlabel('kPCA 1')
ax.set_ylabel('kPCA 2')
ax.set_zlabel('kPCA 3')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import Isomap

isomap = Isomap(n_components=2, n_neighbors=10)
X_iso = isomap.fit_transform(X_test)

plt.figure(figsize=(6, 5))
for cls in [0, 1]:
    plt.scatter(
        X_iso[y_test == cls, 0],
        X_iso[y_test == cls, 1],
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("Isomap 1")
plt.ylabel("Isomap 2")
plt.title("Isomap Projection")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import Isomap
from sklearn.preprocessing import StandardScaler

# Optional: scale using a scaler fitted on training data
# scaler = StandardScaler().fit(X_train)  # fit on training set only
# X_test_scaled = scaler.transform(X_test)
# Use X_test_scaled below if you scaled; otherwise use X_test
X_input = X_test  # or X_test_scaled

# Fit Isomap with 3 components
isomap = Isomap(n_components=3, n_neighbors=10)
X_iso3 = isomap.fit_transform(X_input)  # shape (n_samples, 3)

# Option A: color by y_test using a colormap
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(
    X_iso3[:, 0], X_iso3[:, 1], X_iso3[:, 2],
    c=y_test, cmap='cool', alpha=0.8, s=40, edgecolor='k', linewidth=0.2
)
ax.set_title('3D Isomap (PC1, PC2, PC3)')
ax.set_xlabel('Isomap 1')
ax.set_ylabel('Isomap 2')
ax.set_zlabel('Isomap 3')
plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.1, label='label value')
plt.tight_layout()
plt.show()

# Option B: discrete classes with legend (good for small number of classes)
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
unique_labels = np.unique(y_test)
colors = plt.cm.get_cmap('tab10', len(unique_labels))

for i, lab in enumerate(unique_labels):
    mask = (y_test == lab)
    ax.scatter(
        X_iso3[mask, 0], X_iso3[mask, 1], X_iso3[mask, 2],
        color=colors(i), label=str(lab), s=50, alpha=0.85, edgecolor='k', linewidth=0.2
    )

ax.set_title('3D Isomap by class')
ax.set_xlabel('Isomap 1')
ax.set_ylabel('Isomap 2')
ax.set_zlabel('Isomap 3')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import SpectralEmbedding

se = SpectralEmbedding(n_components=2, n_neighbors=10)
X_se = se.fit_transform(X_test)

plt.figure(figsize=(6, 5))
for cls in [0, 1]:
    plt.scatter(
        X_se[y_test == cls, 0],
        X_se[y_test == cls, 1],
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("SE 1")
plt.ylabel("SE 2")
plt.title("Spectral Embedding")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from scipy.spatial.distance import pdist, squareform
import seaborn as sns

D = squareform(pdist(X_test, metric='cosine'))

same_class = []
diff_class = []

n = len(y_test)
for i in range(n):
    for j in range(i + 1, n):
        if y_test[i] == y_test[j]:
            same_class.append(D[i, j])
        else:
            diff_class.append(D[i, j])

plt.figure(figsize=(6, 4))
sns.kdeplot(same_class, label="Same class", fill=True)
sns.kdeplot(diff_class, label="Different class", fill=True)

plt.xlabel("Cosine Distance")
plt.title("Intra-class vs Inter-class Distances")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

X_train,_, y_train,_ = train_test_split(X, y, test_size=0.99, random_state=42)

In [ ]:
from sklearn.svm import SVC
from sklearn.decomposition import PCA

X_2d = PCA(n_components=2).fit_transform(X_test)

clf = SVC(kernel='rbf')
clf.fit(X_2d, y_test)

xx, yy = np.meshgrid(
    np.linspace(X_2d[:,0].min(), X_2d[:,0].max(), 300),
    np.linspace(X_2d[:,1].min(), X_2d[:,1].max(), 300)
)

Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.2)

for cls in [0, 1]:
    plt.scatter(
        X_2d[y_test == cls, 0],
        X_2d[y_test == cls, 1],
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Decision Boundary (Linear SVM)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.svm import SVC
from sklearn.decomposition import PCA

X_2d = PCA(n_components=2).fit_transform(X_train)

clf = SVC(kernel='rbf')
clf.fit(X_2d, y_train)

xx, yy = np.meshgrid(
    np.linspace(X_2d[:,0].min(), X_2d[:,0].max(), 300),
    np.linspace(X_2d[:,1].min(), X_2d[:,1].max(), 300)
)

Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.2)

for cls in [0, 1]:
    plt.scatter(
        X_2d[y_train == cls, 0],
        X_2d[y_train == cls, 1],
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Decision Boundary (Linear SVM)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import SpectralEmbedding

se = SpectralEmbedding(n_components=2, n_neighbors=10)
X_se = se.fit_transform(X_test)

plt.figure(figsize=(6, 5))
for cls in [0, 1]:
    plt.scatter(
        X_se[y_test == cls, 0],
        X_se[y_test == cls, 1],
        label=f"Class {cls}",
        alpha=0.7
    )

plt.xlabel("SE 1")
plt.ylabel("SE 2")
plt.title("Spectral Embedding")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from scipy.spatial.distance import pdist, squareform
import seaborn as sns

D = squareform(pdist(X_test, metric='cosine'))

same_class = []
diff_class = []

n = len(y_test)
for i in range(n):
    for j in range(i + 1, n):
        if y_test[i] == y_test[j]:
            same_class.append(D[i, j])
        else:
            diff_class.append(D[i, j])

plt.figure(figsize=(6, 4))
sns.kdeplot(same_class, label="Same class", fill=True)
sns.kdeplot(diff_class, label="Different class", fill=True)

plt.xlabel("Cosine Distance")
plt.title("Intra-class vs Inter-class Distances")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
print(X_test.shape)
print(X_test_pca.shape)

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, silhouette_score, normalized_mutual_info_score
import warnings
warnings.filterwarnings('ignore')

def evaluate_clustering(X, labels_true, labels_pred):
    """Evaluate clustering performance"""
    results = {}
    
    # Only calculate ARI if we have true labels
    if labels_true is not None:
        results['ARI'] = adjusted_rand_score(labels_true, labels_pred)
        results['NMI'] = normalized_mutual_info_score(labels_true, labels_pred)
    
    # Calculate silhouette score (doesn't need true labels)
    results['Silhouette'] = silhouette_score(X, labels_pred)
    evaluate_model(labels_true, labels_pred, None, "cf")
    return results

def apply_clustering_techniques(X, y_true=None, n_clusters=2):
    """Apply multiple clustering techniques"""
    
    results = {}
    
    # 1. K-Means Clustering
    print("1. K-Means Clustering")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(X)
    results['KMeans'] = {
        'labels': kmeans_labels,
        'metrics': evaluate_clustering(X, y_true, kmeans_labels)
    }
    print(f"   Centroids: {kmeans.cluster_centers_}")
    print(f"   Metrics: {results['KMeans']['metrics']}")
    
    # 2. Gaussian Mixture Model (Soft Clustering)
    print("\n2. Gaussian Mixture Model")
    gmm = GaussianMixture(n_components=n_clusters, random_state=42)
    gmm_labels = gmm.fit_predict(X)
    results['GMM'] = {
        'labels': gmm_labels,
        'metrics': evaluate_clustering(X, y_true, gmm_labels),
        'probabilities': gmm.predict_proba(X)
    }
    print(f"   Means: {gmm.means_}")
    print(f"   Metrics: {results['GMM']['metrics']}")
    
    # 3. Agglomerative Hierarchical Clustering
    print("\n3. Agglomerative Clustering")
    agg = AgglomerativeClustering(n_clusters=n_clusters)
    agg_labels = agg.fit_predict(X)
    results['Agglomerative'] = {
        'labels': agg_labels,
        'metrics': evaluate_clustering(X, y_true, agg_labels)
    }
    print(f"   Metrics: {results['Agglomerative']['metrics']}")
    
    # 4. DBSCAN (Density-based)
    print("\n4. DBSCAN")
    # Try to find good parameters automatically
    from sklearn.neighbors import NearestNeighbors
    
    # Estimate eps using k-distance graph
    neigh = NearestNeighbors(n_neighbors=5)
    nbrs = neigh.fit(X)
    distances, indices = nbrs.kneighbors(X)
    distances = np.sort(distances[:, -1])
    
    # Use elbow method to find eps
    plt.figure(figsize=(8, 4))
    plt.plot(distances)
    plt.title('k-distance graph for DBSCAN eps estimation')
    plt.xlabel('Points sorted by distance')
    plt.ylabel('5th Nearest Neighbor Distance')
    plt.show()
    
    # Try different eps values
    eps_values = [np.percentile(distances, 50), np.percentile(distances, 75), np.percentile(distances, 90)]
    
    best_dbscan = None
    best_score = -1
    
    for eps in eps_values:
        dbscan = DBSCAN(eps=eps, min_samples=5)
        dbscan_labels = dbscan.fit_predict(X)
        
        # Count unique clusters (excluding noise -1)
        n_clusters_found = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
        
        if n_clusters_found == n_clusters:
            score = silhouette_score(X, dbscan_labels)
            if score > best_score:
                best_score = score
                best_dbscan = dbscan
                best_labels = dbscan_labels
    
    if best_dbscan is not None:
        results['DBSCAN'] = {
            'labels': best_labels,
            'metrics': evaluate_clustering(X, y_true, best_labels),
            'eps': best_dbscan.eps,
            'n_clusters_found': len(set(best_labels)) - (1 if -1 in best_labels else 0)
        }
        print(f"   Best eps: {results['DBSCAN']['eps']}")
        print(f"   Clusters found: {results['DBSCAN']['n_clusters_found']}")
        print(f"   Metrics: {results['DBSCAN']['metrics']}")
    else:
        print("   DBSCAN could not find the desired number of clusters")
        results['DBSCAN'] = None
    
    # 5. Spectral Clustering
    print("\n5. Spectral Clustering")
    spectral = SpectralClustering(n_clusters=n_clusters, random_state=42, affinity='nearest_neighbors')
    spectral_labels = spectral.fit_predict(X)
    results['Spectral'] = {
        'labels': spectral_labels,
        'metrics': evaluate_clustering(X, y_true, spectral_labels)
    }
    print(f"   Metrics: {results['Spectral']['metrics']}")
    
    return results

# Apply clustering techniques
clustering_results = apply_clustering_techniques(X_test, y_test, n_clusters=2)

In [ ]:
def visualize_clustering_results(X, y_true, results, method_names=None):
    """Visualize clustering results"""
    
    if method_names is None:
        method_names = list(results.keys())
    
    n_methods = len(method_names)
    fig, axes = plt.subplots(2, (n_methods + 1) // 2, figsize=(15, 8))
    axes = axes.flatten()
    
    # Plot true labels
    pca = PCA(n_components=4)
    X_pca = pca.fit_transform(X)
    
    axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='viridis', alpha=0.6)
    axes[0].set_title('True Labels')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    
    # Plot clustering results
    for i, method in enumerate(method_names):
        if results[method] is not None:
            axes[i+1].scatter(X_pca[:, 0], X_pca[:, 1], 
                             c=results[method]['labels'], cmap='viridis', alpha=0.6)
            axes[i+1].set_title(f'{method}\nARI: {results[method]["metrics"].get("ARI", "N/A"):.3f}')
            axes[i+1].set_xlabel('PC1')
            axes[i+1].set_ylabel('PC2')
    
    plt.tight_layout()
    plt.show()

# Visualize results
visualize_clustering_results(X_test, y_test, clustering_results)

In [ ]:
def adaptive_clustering_with_shift_correction(X_test, n_clusters=2):
    """
    Adaptive approach that handles centroid shift by:
    1. Finding clusters in test data
    2. Aligning with training distribution if reference is available
    """
    
    # Method 1: Use Gaussian Mixture for soft clustering
    gmm = GaussianMixture(n_components=n_clusters, random_state=42)
    gmm_labels = gmm.fit_predict(X_test)
    gmm_probs = gmm.predict_proba(X_test)
    
    # Method 2: Use KMeans with initialization from test data statistics
    # Scale the data first
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_test)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(X_scaled)
    
    # Method 3: Ensemble clustering (combine multiple methods)
    from scipy.stats import mode
    
    # Collect labels from different methods
    all_labels = []
    
    # KMeans
    all_labels.append(kmeans_labels)
    
    # GMM
    all_labels.append(gmm_labels)
    
    # Agglomerative
    agg = AgglomerativeClustering(n_clusters=n_clusters)
    agg_labels = agg.fit_predict(X_scaled)
    all_labels.append(agg_labels)
    
    # Ensemble by majority voting
    all_labels_array = np.array(all_labels)
    ensemble_labels, _ = mode(all_labels_array, axis=0)
    ensemble_labels = ensemble_labels.flatten()
    
    return {
        'GMM': {'labels': gmm_labels, 'probs': gmm_probs},
        'KMeans': {'labels': kmeans_labels},
        'Ensemble': {'labels': ensemble_labels},
        'scaled_data': X_scaled,
        'scaler': scaler
    }

# Apply adaptive clustering
adaptive_results = adaptive_clustering_with_shift_correction(X_test)

# Compare with true labels
print("Adaptive Clustering Performance:")
print("=" * 50)

for method in ['GMM', 'KMeans', 'Ensemble']:
    if method in adaptive_results:
        labels_pred = adaptive_results[method]['labels']
        ari = adjusted_rand_score(y_test, labels_pred)
        acc = np.mean(labels_pred == y_test)
        # Handle label swapping (clustering might reverse 0/1)
        acc = max(acc, 1 - acc)  # Take max because labels might be swapped
        evaluate_model(y_test, labels_pred, None, method)
        print(f"{method}:")
        print(f"  Adjusted Rand Index: {ari:.3f}")
        print(f"  Accuracy: {acc:.3f}")
        print(f"  Cluster sizes: {np.bincount(labels_pred)}")
        print()

In [ ]:
def density_based_clustering_advanced(X, n_clusters=2):
    """Advanced density-based clustering with automatic parameter tuning"""
    
    from sklearn.neighbors import NearestNeighbors
    from sklearn.cluster import OPTICS
    
    # Approach 1: OPTICS (more robust than DBSCAN)
    optics = OPTICS(min_samples=10, xi=0.05, min_cluster_size=0.1)
    optics_labels = optics.fit_predict(X)
    
    # Count clusters found (excluding noise)
    unique_labels = set(optics_labels)
    n_clusters_found = len(unique_labels) - (1 if -1 in unique_labels else 0)
    
    print(f"OPTICS found {n_clusters_found} clusters")
    
    # Approach 2: HDBSCAN (if installed)
    try:
        import hdbscan
        hdb = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5)
        hdb_labels = hdb.fit_predict(X)
        
        unique_labels_hdb = set(hdb_labels)
        n_clusters_hdb = len(unique_labels_hdb) - (1 if -1 in unique_labels_hdb else 0)
        print(f"HDBSCAN found {n_clusters_hdb} clusters")
        
        return {
            'OPTICS': optics_labels,
            'HDBSCAN': hdb_labels if n_clusters_hdb == n_clusters else None
        }
    except:
        print("HDBSCAN not installed. Install with: pip install hdbscan")
        return {'OPTICS': optics_labels}
density_based_clustering_advanced(xdf)

In [ ]:
def get_best_clustering_predictions(X_test, y_test=None):
    """
    Get the best clustering predictions for test data
    Returns predicted labels and confidence scores
    """
    
    # Scale the data first
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_test)
    
    # Try multiple methods
    methods = {}
    
    # 1. Gaussian Mixture (with probability estimates)
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm_labels = gmm.fit_predict(X_scaled)
    gmm_probs = gmm.predict_proba(X_scaled)
    gmm_confidence = np.max(gmm_probs, axis=1)
    
    methods['GMM'] = {
        'labels': gmm_labels,
        'confidence': gmm_confidence,
        'probs': gmm_probs
    }
    
    # 2. KMeans
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(X_scaled)
    # Estimate confidence based on distance to centroids
    distances = kmeans.transform(X_scaled)
    kmeans_confidence = 1 / (1 + np.min(distances, axis=1))
    
    methods['KMeans'] = {
        'labels': kmeans_labels,
        'confidence': kmeans_confidence
    }
    
    # Evaluate if true labels are available
    if y_test is not None:
        for name, result in methods.items():
            # Handle potential label swapping
            labels = result['labels']
            acc1 = np.mean(labels == y_test)
            acc2 = np.mean(labels != y_test)  # Swapped labels
            accuracy = max(acc1, acc2)
            
            ari = adjusted_rand_score(y_test, labels)
            print(f"{name}: Accuracy={accuracy:.3f}, ARI={ari:.3f}")
            evaluate_model(y_test,labels,None,"best")
    # Choose the best method based on silhouette score
    best_method = None
    best_score = -1
    
    for name, result in methods.items():
        score = silhouette_score(X_scaled, result['labels'])
        if score > best_score:
            best_score = score
            best_method = name
    
    print(f"\nBest method: {best_method} (Silhouette score: {best_score:.3f})")
    
    return methods[best_method]

# Get best clustering predictions
best_predictions = get_best_clustering_predictions(X_test, y_test)

# Use the predictions
predicted_labels = best_predictions['labels']
confidence_scores = best_predictions.get('confidence', None)

print(f"\nPredicted class distribution:")
print(f"Class 0: {np.sum(predicted_labels == 0)} samples")
print(f"Class 1: {np.sum(predicted_labels == 1)} samples")

In [ ]:
def evaluate_model(y_true, y_pred, y_scores, model_name):
    """Comprehensive evaluation function"""
    print(f"\n{model_name} Performance:")
    print("-" * 30)
    
    # Basic metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary')
    recall = recall_score(y_true, y_pred, average='binary')
    f1 = f1_score(y_true, y_pred, average='binary')
    
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # ROC-AUC if we have probability scores
    if y_scores is not None:
        try:
            roc_auc = roc_auc_score(y_true, y_scores)
            print(f"ROC-AUC:   {roc_auc:.4f}")
        except:
            print("ROC-AUC:   Not available")
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"[[TN={cm[0,0]}  FP={cm[0,1]}]")
    print(f" [FN={cm[1,0]}  TP={cm[1,1]}]]")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1']))
        
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=range(cm.shape[0]),
        yticklabels=range(cm.shape[0])
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()
    
        # Flip predictions (assuming binary classification: 0/1)
    # labels_pred_flipped = 1 - y_pred
    # # Confusion matrix for flipped predictions
    # cm_flipped = confusion_matrix(y_test, labels_pred_flipped)

    # plt.figure(figsize=(8, 6))
    # sns.heatmap(
    #     cm_flipped,
    #     annot=True,
    #     fmt="d",
    #     cmap="Oranges",
    #     xticklabels=range(cm_flipped.shape[0]),
    #     yticklabels=range(cm_flipped.shape[0])
    # )
    # plt.xlabel("Predicted")
    # plt.ylabel("True")
    # plt.title("Confusion Matrix")
    # plt.show()

# Logistic regression

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve, auc)
# 1. Prepare the data
print("Preparing data...")
X_train,_, y_train,_ = train_test_split(X, y, test_size=0.9, random_state=42)
# extract first 18 features
X_train = X_train[:, :25].copy() 
X_test = X_test[:, :25].copy()
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}") 
print(f"Class distribution in training set: {np.bincount(y_train)}")
print(f"Class distribution in testing set: {np.bincount(y_test)}")

# 2. Logistic Regression
print("\n" + "="*50)
print("LOGISTIC REGRESSION")
print("="*50)

# Train Logistic Regression
logreg = LogisticRegression(
    penalty='l2',           # Regularization
    C=1.0,                  # Inverse regularization strength
    solver='lbfgs',         # Good for multiclass
    max_iter=1000,
    random_state=42
)
# Show warnings explicitly\
logreg.fit(X_train, y_train)

# Predictions
y_pred_logreg = logreg.predict(X_test)
y_pred_proba_logreg = logreg.predict_proba(X_test)[:, 1]
import pandas as pd

# Assuming y_pred_proba_logreg is already defined
df = pd.DataFrame({'y_pred_proba': y_pred_proba_logreg})

# Save to CSV
df.to_csv('y_pred_proba_logreg.csv', index=False)

# Cross-validation
cv_scores = cross_val_score(logreg, X_train, y_train, cv=StratifiedKFold(n_splits=2, shuffle=True, random_state=42))
print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# 3. Linear SVM
print("\n" + "="*50)
print("LINEAR SVM")
print("="*50)

# Train Linear SVM
svm_linear = LinearSVC(
    penalty='l2',
    C=1.0,
    loss='squared_hinge',
    max_iter=65,
    dual = False,
    random_state=42
)

svm_linear.fit(X_train, y_train)

# Predictions
y_pred_svm = svm_linear.predict(X_test)

# For SVM, we need to get decision function scores for ROC
if hasattr(svm_linear, 'decision_function'):
    y_scores_svm = svm_linear.decision_function(X_test)
else:
    # Fallback: use predict for scores
    y_scores_svm = svm_linear.predict(X_test)

# Cross-validation for SVM
cv_scores_svm = cross_val_score(svm_linear, X_train, y_train, cv=StratifiedKFold(n_splits=2, shuffle=True, random_state=42))
print(f"Cross-validation accuracy: {cv_scores_svm.mean():.4f} (+/- {cv_scores_svm.std() * 2:.4f})")

# 4. Evaluation Metrics
def evaluate_model(y_true, y_pred, y_scores, model_name):
    """Comprehensive evaluation function"""
    print(f"\n{model_name} Performance:")
    print("-" * 30)
    
    # Basic metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary')
    recall = recall_score(y_true, y_pred, average='binary')
    f1 = f1_score(y_true, y_pred, average='binary')
    
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # ROC-AUC if we have probability scores
    if y_scores is not None:
        try:
            roc_auc = roc_auc_score(y_true, y_scores)
            print(f"ROC-AUC:   {roc_auc:.4f}")
        except:
            print("ROC-AUC:   Not available")
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"[[TN={cm[0,0]}  FP={cm[0,1]}]")
    print(f" [FN={cm[1,0]}  TP={cm[1,1]}]]")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1']))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cm': cm
    }

# Evaluate both models
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

results_logreg = evaluate_model(y_test, y_pred_logreg, y_pred_proba_logreg, "Logistic Regression")
results_svm = evaluate_model(y_test, y_pred_svm, y_scores_svm, "Linear SVM")

# 5. Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 5.1 ROC Curves
ax = axes[0, 0]
if y_pred_proba_logreg is not None:
    fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_logreg)
    roc_auc_lr = auc(fpr_lr, tpr_lr)
    ax.plot(fpr_lr, tpr_lr, color='blue', lw=2, 
            label=f'Logistic Regression (AUC = {roc_auc_lr:.3f})')

if y_scores_svm is not None:
    try:
        fpr_svm, tpr_svm, _ = roc_curve(y_test, y_scores_svm)
        roc_auc_svm = auc(fpr_svm, tpr_svm)
        ax.plot(fpr_svm, tpr_svm, color='red', lw=2, 
                label=f'Linear SVM (AUC = {roc_auc_svm:.3f})')
    except:
        pass

ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

# 5.2 Confusion Matrices
# Logistic Regression Confusion Matrix
ax = axes[0, 1]
sns.heatmap(results_logreg['cm'], annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Logistic Regression Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticklabels(['Class 0', 'Class 1'])
ax.set_yticklabels(['Class 0', 'Class 1'])

# SVM Confusion Matrix
ax = axes[0, 2]
sns.heatmap(results_svm['cm'], annot=True, fmt='d', cmap='Reds', ax=ax)
ax.set_title('Linear SVM Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticklabels(['Class 0', 'Class 1'])
ax.set_yticklabels(['Class 0', 'Class 1'])

# 5.3 Feature Importance (Logistic Regression)
ax = axes[1, 0]
if hasattr(logreg, 'coef_'):
    feature_importance = pd.DataFrame({
        'importance': np.abs(logreg.coef_[0])
    }).sort_values('importance', ascending=False).head(15)
    
    ax.barh(range(len(feature_importance)), feature_importance['importance'])
    ax.set_yticks(range(len(feature_importance)))
    ax.invert_yaxis()
    ax.set_xlabel('Absolute Coefficient Value')
    ax.set_title('Top 15 Features (Logistic Regression)')
    ax.grid(True, alpha=0.3, axis='x')

# 5.4 Model Comparison Bar Plot
ax = axes[1, 1]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
logreg_metrics = [results_logreg['accuracy'], results_logreg['precision'], 
                  results_logreg['recall'], results_logreg['f1']]
svm_metrics = [results_svm['accuracy'], results_svm['precision'], 
               results_svm['recall'], results_svm['f1']]

x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, logreg_metrics, width, label='Logistic Regression', color='blue', alpha=0.7)
ax.bar(x + width/2, svm_metrics, width, label='Linear SVM', color='red', alpha=0.7)

ax.set_xlabel('Metrics')
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim([0, 1.05])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 5.5 Learning Curves (Simplified)
ax = axes[1, 2]
# Train on increasing data sizes
train_sizes = np.linspace(0.1, 1.0, 10)
train_scores_logreg = []
test_scores_logreg = []

for size in train_sizes:
    n_samples = int(size * len(X_train))
    X_subset = X_train[:n_samples]
    y_subset = y_train[:n_samples]
    
    logreg_temp = LogisticRegression(max_iter=1000, random_state=42)
    logreg_temp.fit(X_subset, y_subset)
    
    train_scores_logreg.append(logreg_temp.score(X_subset, y_subset))
    test_scores_logreg.append(logreg_temp.score(X_test, y_test))

ax.plot(train_sizes * 100, train_scores_logreg, 'o-', color='blue', label='LogReg Train')
ax.plot(train_sizes * 100, test_scores_logreg, 'o-', color='red', label='LogReg Test')
ax.set_xlabel('Training Set Size (%)')
ax.set_ylabel('Accuracy')
ax.set_title('Learning Curve (Logistic Regression)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Detailed Summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'CV Mean', 'CV Std'],
    'Logistic Regression': [
        results_logreg['accuracy'],
        results_logreg['precision'],
        results_logreg['recall'],
        results_logreg['f1'],
        cv_scores.mean(),
        cv_scores.std()
    ],
    'Linear SVM': [
        results_svm['accuracy'],
        results_svm['precision'],
        results_svm['recall'],
        results_svm['f1'],
        cv_scores_svm.mean(),
        cv_scores_svm.std()
    ]
})

print(comparison_df.to_string(index=False))

# Determine which model performed better
if results_logreg['accuracy'] > results_svm['accuracy']:
    print(f"\n✓ Logistic Regression performed better by {results_logreg['accuracy'] - results_svm['accuracy']:.4f} accuracy")
    best_model = logreg
    best_model_name = "Logistic Regression"
else:
    print(f"\n✓ Linear SVM performed better by {results_svm['accuracy'] - results_logreg['accuracy']:.4f} accuracy")
    best_model = svm_linear
    best_model_name = "Linear SVM"

print(f"\nBest Model: {best_model_name}")
print(f"Test Accuracy: {max(results_logreg['accuracy'], results_svm['accuracy']):.4f}")

# 7. Optional: Save models
print("\nSaving models...")
import joblib
joblib.dump(logreg, 'logistic_regression_model.pkl')
joblib.dump(svm_linear, 'linear_svm_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("Models saved as 'logistic_regression_model.pkl', 'linear_svm_model.pkl', and 'scaler.pkl'")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, auc
)
# Training function for Decision Trees with different configurations
def train_decision_trees(X_train, y_train, X_test, y_test):
    """
    Train multiple Decision Tree models with different configurations
    """
    models = {}
    
    # Define different Decision Tree configurations
    dt_configs = {
        'dt_default': DecisionTreeClassifier(random_state=42),
        'dt_gini_depth5': DecisionTreeClassifier(
            criterion='gini',
            max_depth=5,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=42
        ),
        'dt_entropy_depth10': DecisionTreeClassifier(
            criterion='entropy',
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            random_state=42
        ),
        'dt_pruned': DecisionTreeClassifier(
            criterion='gini',
            max_depth=None,  # Let tree grow
            min_samples_split=2,
            min_samples_leaf=1,
            max_features='sqrt',  # Consider sqrt of features at each split
            ccp_alpha=0.01,  # Cost complexity pruning
            random_state=42
        )
    }
    
    # Train each model
    for model_name, model in dt_configs.items():
        print(f"Training {model_name}...")
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(XT_scaled)
        y_scores = model.predict_proba(XT_scaled)[:, 1]
        submission = pd.DataFrame({
            "ID": tdf["ID"],
            "label": y_pred,
            "scores": y_scores
        })
        
        # Save the prediction to a CSV file with model name in the file name
        submission_filename = f"submission_{model_name}.csv"
        submission.to_csv(submission_filename, index=False)
        
        print(f"{submission_filename} created successfully!")
        # Assuming y_pred_proba_logreg is already defined
        df = pd.DataFrame({'y_pred_proba': y_scores})
        
        # Save to CSV
        df.to_csv(f'y_score_{model_name}.csv', index=False)

        # Evaluate
        #evaluate_model(y_test, y_pred, y_scores, model_name)
        
        # Store model
        models[model_name] = model
        
        print("-" * 50)
    
    return models

# Evaluation function
def evaluate_model(y_true, y_pred, y_scores, model_name):
    """
    Comprehensive evaluation of classification model
    """
    print(f"\n{'='*60}")
    print(f"EVALUATION RESULTS FOR: {model_name}")
    print(f"{'='*60}")
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # ROC-AUC (if binary classification)
    if len(np.unique(y_true)) == 2:
        try:
            roc_auc = roc_auc_score(y_true, y_scores)
            print(f"ROC-AUC Score: {roc_auc:.4f}")
        except:
            roc_auc = None
            print("Could not calculate ROC-AUC score")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Print metrics
    print(f"\nBasic Metrics:")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # Print confusion matrix
    print(f"\nConfusion Matrix:")
    print(f"True Negatives: {cm[0,0]}")
    print(f"False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]}")
    print(f"True Positives: {cm[1,1]}")
    
    # Classification report
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1']))
    
    # Visualizations
    plot_evaluation_metrics(y_true, y_pred, y_scores, model_name, cm)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc if 'roc_auc' in locals() else None,
        'confusion_matrix': cm
    }

def plot_evaluation_metrics(y_true, y_pred, y_scores, model_name, cm):
    """
    Plot evaluation metrics
    """
    fig, axes = plt.subplots(2, 2, figsize=(10, 12))
    fig.suptitle(f'Model Evaluation: {model_name}', fontsize=16, fontweight='bold')
    
    # 1. Confusion Matrix Heatmap
    ax1 = axes[0, 0]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Predicted 0', 'Predicted 1'],
                yticklabels=['Actual 0', 'Actual 1'], ax=ax1)
    ax1.set_title('Confusion Matrix')
    ax1.set_ylabel('True Label')
    ax1.set_xlabel('Predicted Label')
    
    # 2. ROC Curve (if binary classification)
    ax2 = axes[0, 1]
    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        
        ax2.plot(fpr, tpr, color='darkorange', lw=2, 
                label=f'ROC curve (AUC = {roc_auc:.2f})')
        ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        ax2.set_xlim([0.0, 1.0])
        ax2.set_ylim([0.0, 1.05])
        ax2.set_xlabel('False Positive Rate')
        ax2.set_ylabel('True Positive Rate')
        ax2.set_title('ROC Curve')
        ax2.legend(loc="lower right")
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, 'ROC Curve only for binary classification',
                horizontalalignment='center', verticalalignment='center',
                transform=ax2.transAxes)
        ax2.set_title('ROC Curve (Not available)')
    
    # 3. Metrics Bar Chart
    ax3 = axes[1, 0]
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    values = [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, zero_division=0),
        recall_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0)
    ]
    
    bars = ax3.bar(metrics, values, color=['blue', 'green', 'orange', 'red'])
    ax3.set_ylim([0, 1])
    ax3.set_title('Performance Metrics')
    ax3.set_ylabel('Score')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{value:.3f}', ha='center', va='bottom')
    
    # 4. Feature Importance (if using tree models)
    ax4 = axes[1, 1]
    ax4.text(0.5, 0.5, 'Feature importance would be shown here\nfor tree-based models',
            horizontalalignment='center', verticalalignment='center',
            transform=ax4.transAxes)
    ax4.set_title('Feature Importance')
    
    plt.tight_layout()
    plt.show()

# Main training and evaluation
def main_training_pipeline():
    """
    Complete training and evaluation pipeline
    """

    print("Starting Decision Tree Training Pipeline...")
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")
    print(f"Class distribution in training: {np.bincount(y_train)}")
    print(f"Class distribution in test: {np.bincount(y_test)}")
    
    # Train Decision Trees
    print("\n" + "="*60)
    print("TRAINING DECISION TREES")
    print("="*60)
    dt_models = train_decision_trees(X_train, y_train, X_test, y_test)
    
    # You can also add model saving functionality
    # save_models(dt_models)
    
    return dt_models
main_training_pipeline()

In [ ]:
import pandas as pd

# Split the string back into a list
values_str = """
0.2843807199511897,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.1763001974983541,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.1763001974983541,0.1777618395601052,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.1763001974983541,0.9985877528295866,0.9985877528295866,0.2843807199511897,0.1777618395601052,0.9985877528295866,0.1763001974983541,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.9985877528295866,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.3597043818650466,0.2843807199511897,0.8224813924866671,0.1777618395601052,0.1777618395601052,0.1777618395601052,0.9985877528295866,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.1777618395601052,0.2843807199511897,0.3597043818650466,0.8224813924866671,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.1777618395601052,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.1763001974983541,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.1777618395601052,0.9985877528295866,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.3597043818650466,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.1777618395601052,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.1777618395601052,0.8224813924866671,0.1777618395601052,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.3597043818650466,0.1763001974983541,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.3597043818650466,0.8224813924866671,0.9985877528295866,0.1777618395601052,0.1777618395601052,0.3597043818650466,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.1777618395601052,0.9985877528295866,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.3597043818650466,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.3597043818650466,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.3597043818650466,0.9985877528295866,0.9985877528295866,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.3597043818650466,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.1763001974983541,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.9985877528295866,0.3597043818650466,0.1777618395601052,0.1777618395601052,0.8224813924866671,0.9985877528295866,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.1777618395601052,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.3597043818650466,0.9985877528295866,0.8224813924866671,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.3597043818650466,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.1777618395601052,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.1763001974983541,0.1763001974983541,0.9985877528295866,0.3597043818650466,0.3597043818650466,0.1777618395601052,0.2843807199511897,0.8224813924866671,0.1777618395601052,0.2843807199511897,0.8224813924866671,0.1763001974983541,0.9985877528295866,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.1763001974983541,0.1763001974983541,0.8224813924866671,0.8224813924866671,0.1777618395601052,0.1777618395601052,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.1763001974983541,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.1777618395601052,0.9985877528295866,0.1763001974983541,0.8224813924866671,0.1777618395601052,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.3597043818650466,0.1777618395601052,0.1777618395601052,0.1777618395601052,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.1777618395601052,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.3597043818650466,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.9985877528295866,0.1763001974983541,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.3597043818650466,0.1777618395601052,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.1763001974983541,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.1763001974983541,0.3597043818650466,0.1777618395601052,0.9985877528295866,0.9985877528295866,0.3597043818650466,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.8224813924866671,0.3597043818650466,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.1777618395601052,0.9985877528295866,0.2843807199511897,0.1763001974983541,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.3597043818650466,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.3597043818650466,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.1777618395601052,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.1763001974983541,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.1777618395601052,0.3597043818650466,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.1777618395601052,0.1777618395601052,0.8224813924866671,0.9985877528295866,0.1777618395601052,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.1763001974983541,0.3597043818650466,0.2843807199511897,0.1777618395601052,0.1763001974983541,0.1777618395601052,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.1763001974983541,0.9985877528295866,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.3597043818650466,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.1777618395601052,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.9985877528295866,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.1763001974983541,0.3597043818650466,0.9985877528295866,0.2843807199511897,0.1763001974983541,0.9985877528295866,0.2843807199511897,0.1763001974983541,0.3597043818650466,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.3597043818650466,0.1763001974983541,0.9985877528295866,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.1763001974983541,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.1777618395601052,0.3597043818650466,0.1777618395601052,0.1777618395601052,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.1763001974983541,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.9985877528295866,0.2843807199511897,0.3597043818650466,0.1777618395601052,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.1763001974983541,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.3597043818650466,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.1777618395601052,0.9985877528295866,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.1777618395601052,0.8224813924866671,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.1777618395601052,0.1777618395601052,0.1777618395601052,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.1763001974983541,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.9985877528295866,0.1763001974983541,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.1777618395601052,0.3597043818650466,0.2843807199511897,0.1763001974983541,0.9985877528295866,0.1763001974983541,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.3597043818650466,0.2843807199511897,0.1763001974983541,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.1777618395601052,0.1763001974983541,0.2843807199511897,0.1763001974983541,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.9985877528295866,0.9985877528295866,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.8224813924866671,0.1763001974983541,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.8224813924866671,0.8224813924866671,0.9985877528295866,0.9985877528295866,0.2843807199511897,0.1777618395601052,0.9985877528295866,0.9985877528295866,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.9985877528295866,0.1763001974983541,0.8224813924866671,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.8224813924866671,0.9985877528295866,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.3597043818650466,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.1777618395601052,0.8224813924866671,0.1763001974983541,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.1777618395601052,0.3597043818650466,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.3597043818650466,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.1763001974983541,0.9985877528295866,0.9985877528295866,0.1777618395601052,0.9985877528295866,0.3597043818650466,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.8224813924866671,0.1777618395601052,0.1777618395601052,0.9985877528295866,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.1763001974983541,0.2843807199511897,0.1763001974983541,0.9985877528295866,0.2843807199511897,0.1777618395601052,0.9985877528295866,0.2843807199511897,0.3597043818650466,0.1777618395601052,0.9985877528295866,0.3597043818650466,0.3597043818650466,0.1763001974983541,0.8224813924866671,0.3597043818650466,0.9985877528295866,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.2843807199511897,0.9985877528295866,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.2843807199511897,0.9985877528295866,0.8224813924866671,0.1777618395601052,0.1777618395601052,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.9985877528295866,0.2843807199511897,0.8224813924866671,0.9985877528295866,0.3597043818650466,0.2843807199511897,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.1777618395601052,0.2843807199511897,0.2843807199511897,0.1763001974983541,0.8224813924866671,0.1763001974983541,0.8224813924866671,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.2843807199511897,0.8224813924866671,0.8224813924866671,0.2843807199511897,0.2843807199511897,0.3597043818650466,0.2843807199511897"""
values_list = values_str.split(',')

# Convert list into DataFrame (single column)
df_re  = pd.DataFrame({'values': values_list})
df_re.to_csv(f'featurescore.csv', index=False)
print(df_re.head())


In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of dt_score values
plt.figure(figsize=(8, 6))
plt.hist(df['dt_score'], bins=20, color='skyblue', edgecolor='black')

plt.title('Distribution of dt_score')
plt.xlabel('dt_score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of dt_score values
plt.figure(figsize=(8, 6))
plt.hist(df['featurescore'], bins=20, color='skyblue', edgecolor='black')

plt.title('Distribution of dt_score')
plt.xlabel('dt_score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score

# Load
df_feature = pd.read_csv('/kaggle/working/featurescore.csv')
df_dt = pd.read_csv('/kaggle/working/y_score_dt_pruned.csv')

df = pd.DataFrame()
df['featurescore'] = df_feature['values']
df['dt_score'] = df_dt['y_pred_proba']
def ensemble(df):
    
    pred = np.zeros(len(df))  # default 0
    
    mask = df['dt_score'] >= 0.5
    pred[mask] = (df.loc[mask, 'featurescore'] >= 0.5).astype(int)
    
    return pred


# Final predictions using best threshold
df['final_prediction'] = ensemble(df)

df[['final_prediction']].to_csv('/kaggle/working/ensemble_prediction.csv', index=False)

df.head()

In [ ]:
# Get the column as a list
y_pred = df['final_prediction'].tolist()
# y_proba = df
# print(y_pred[:10])  # preview first 10 values


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)

def evaluate_model(y_test, y_pred, y_proba=None):
    """
    y_test  : actual labels
    y_pred  : predicted labels (0/1)
    y_proba : predicted probabilities (optional, for ROC & PR curve)
    """
    
    print("========== Classification Report ==========\n")
    print(classification_report(y_test, y_pred))
    
    print("Accuracy:", accuracy_score(y_test, y_pred))
    
    # ROC-AUC
    if y_proba is not None:
        print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    
    # ROC Curve
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.figure(figsize=(6,5))
        plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba):.4f}")
        plt.plot([0,1], [0,1], 'r--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend()
        plt.show()
        
        # Precision-Recall Curve
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        pr_auc = auc(recall, precision)
        
        plt.figure(figsize=(6,5))
        plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.4f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")
        plt.legend()
        plt.show()
evaluate_model(y_test, y_pred)

In [ ]:
XT_scaled = scaler.transform(Xf_test)
# Predict labels (binary classification)
XT_scaled = XT_scaled[:, :25].copy()
pred_labels = logreg.predict(XT_scaled)
pred_scores = logreg.predict_proba(XT_scaled)[:, 1]
# Create submission DataFrame
submission = pd.DataFrame({
    "ID": tdf["ID"],
    "label": pred_labels,
    "scores": pred_scores
})

# Save the prediction to a CSV file with model name in the file name
submission_filename = f"submission_logr.csv"
submission.to_csv(submission_filename, index=False)

print(f"{submission_filename} created successfully!")

In [ ]:

s = pd.read_csv("/kaggle/working/submission_dt_entropy_depth10.csv")
s.info()
s.head(5)

In [ ]:

finaltestdf = pd.read_parquet("/kaggle/working/finaltest_features_all.parquet")
finaltestdf.info()
finaltestdf.head()
tdf = finaltestdf

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
import joblib
import time
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings('ignore')

# Training function for Random Forest with different configurations
def train_random_forests(X_train, y_train, X_test, y_test):
    """
    Train multiple Random Forest models with different configurations
    """
    models = {}
    results = {}
    
    # Define different Random Forest configurations
    rf_configs = { 
        'rf_large' :  RandomForestClassifier(
                        n_estimators=600,
                        max_depth=10,
                        min_samples_leaf=5,
                        max_features="sqrt",
                        n_jobs=-1,
                        random_state=42
                    ),

        
        'rf_tuned' : RandomForestClassifier(
                    n_estimators=200,
                    min_samples_split=15,
                    min_samples_leaf=15,
                    max_features=0.5,  # Use 50% of features
                    bootstrap=True,
                    max_depth=8,
                    random_state=42,
                    n_jobs=-1
                ),
        'rf_basic': RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features='sqrt',
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ),

        
        'rf_fast': RandomForestClassifier(
            n_estimators=50,
            max_depth=8,
            min_samples_split=20,
            min_samples_leaf=10,
            max_features=0.5,  # Use 50% of features
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )
    }
    
    print("Training Random Forest models...")
    print("="*60)
    
    # Train each model
    for model_name, model in rf_configs.items():
        print(f"\nTraining {model_name}...")
        start_time = time.time()
        
        # Train model
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        # Make predictions
        y_pred = model.predict(X_test)
        y_scores = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
        
        # Evaluate
        eval_results = evaluate_model(y_test, y_pred, y_scores, model_name)
        eval_results['training_time'] = training_time
        
        # Store results
        results[model_name] = eval_results
        models[model_name] = model
        
        # Print model info
        print(f"Training time: {training_time:.2f} seconds")
        print(f"Number of trees: {model.n_estimators}")
        print(f"Max depth: {model.max_depth}")
        print("-" * 50)
    
    return models, results

# Hyperparameter tuning for Random Forest
def tune_random_forest(X_train, y_train, X_test, y_test, method='random', cv_folds=3):
    """
    Perform hyperparameter tuning for Random Forest
    """
    print("\n" + "="*60)
    print("RANDOM FOREST HYPERPARAMETER TUNING")
    print("="*60)
    
    # Base model
    rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    
    # Define parameter grid
    param_grid = {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf': [1, 2, 5, 10],
        'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
        'bootstrap': [True, False],
        'class_weight': [None, 'balanced']
    }
    
    # For RandomizedSearchCV - broader ranges
    param_distributions = {
        'n_estimators': randint(50, 500),
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'max_features': uniform(0.1, 0.9),
        'bootstrap': [True, False]
    }
    
    if method == 'grid':
        print("Performing Grid Search CV...")
        search = GridSearchCV(
            estimator=rf,
            param_grid=param_grid,
            cv=cv_folds,
            scoring='roc_auc',
            n_jobs=-1,
            verbose=1
        )
    else:  # random search
        print("Performing Randomized Search CV...")
        search = RandomizedSearchCV(
            estimator=rf,
            param_distributions=param_distributions,
            n_iter=20,  # Number of parameter settings to sample
            cv=cv_folds,
            scoring='roc_auc',
            random_state=42,
            n_jobs=-1,
            verbose=1
        )
    
    # Perform search
    start_time = time.time()
    search.fit(X_train, y_train)
    tuning_time = time.time() - start_time
    
    print(f"\nTuning completed in {tuning_time:.2f} seconds")
    print(f"Best parameters: {search.best_params_}")
    print(f"Best CV score: {search.best_score_:.4f}")
    
    # Train final model with best parameters
    best_rf = search.best_estimator_
    
    # Evaluate on test set
    y_pred = best_rf.predict(X_test)
    y_scores = best_rf.predict_proba(X_test)[:, 1]
    
    results = evaluate_model(y_test, y_pred, y_scores, "Tuned Random Forest")
    results['best_params'] = search.best_params_
    results['tuning_time'] = tuning_time
    
    return best_rf, results, search

# Feature importance analysis
def analyze_feature_importance(model, feature_names, top_n=10):
    """
    Analyze and visualize feature importance
    """
    if hasattr(model, 'feature_importances_'):
        # Get feature importances
        importances = model.feature_importances_
        
        # Create DataFrame
        feature_importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importances
        }).sort_values('importance', ascending=False)
        
        print(f"\n{'='*60}")
        print(f"FEATURE IMPORTANCE ANALYSIS (Top {top_n})")
        print(f"{'='*60}")
        print(feature_importance_df.head(top_n).to_string(index=False))
        
        # Plot feature importance
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        
        # Horizontal bar chart
        top_features = feature_importance_df.head(top_n)
        axes[0].barh(range(len(top_features)), top_features['importance'].values[::-1])
        axes[0].set_yticks(range(len(top_features)))
        axes[0].set_yticklabels(top_features['feature'].values[::-1])
        axes[0].set_xlabel('Importance')
        axes[0].set_title(f'Top {top_n} Feature Importances')
        axes[0].grid(True, alpha=0.3, axis='x')
        
        # Cumulative importance plot
        cumulative_importance = np.cumsum(feature_importance_df['importance'].values)
        axes[1].plot(range(1, len(cumulative_importance) + 1), cumulative_importance, 
                    marker='o', linewidth=2)
        axes[1].axhline(y=0.8, color='r', linestyle='--', alpha=0.7, label='80% importance')
        axes[1].axhline(y=0.9, color='g', linestyle='--', alpha=0.7, label='90% importance')
        axes[1].set_xlabel('Number of Features')
        axes[1].set_ylabel('Cumulative Importance')
        axes[1].set_title('Cumulative Feature Importance')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Find number of features for 80% and 90% importance
        idx_80 = np.where(cumulative_importance >= 0.8)[0][0] + 1
        idx_90 = np.where(cumulative_importance >= 0.9)[0][0] + 1
        
        print(f"\nFeature Reduction Insights:")
        print(f"Top {idx_80} features explain 80% of importance")
        print(f"Top {idx_90} features explain 90% of importance")
        
        return feature_importance_df, idx_80, idx_90
    
    else:
        print("Model does not have feature_importances_ attribute")
        return None, None, None

# Ensemble of Random Forests with different seeds
def create_ensemble_rf(X_train, y_train, X_test, y_test, n_models=5):
    """
    Create an ensemble of Random Forests with different random seeds
    """
    print(f"\n{'='*60}")
    print(f"CREATING ENSEMBLE OF {n_models} RANDOM FORESTS")
    print(f"{'='*60}")
    
    models = []
    predictions = []
    probabilities = []
    
    for i in range(n_models):
        print(f"Training model {i+1}/{n_models}...")
        
        # Create RF with different random seed
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=15,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features='sqrt',
            bootstrap=True,
            random_state=42 + i,  # Different seed for each model
            n_jobs=-1
        )
        
        rf.fit(X_train, y_train)
        models.append(rf)
        
        # Get predictions
        pred = rf.predict(X_test)
        proba = rf.predict_proba(X_test)[:, 1]
        
        predictions.append(pred)
        probabilities.append(proba)
    
    # Ensemble predictions (voting)
    predictions_array = np.array(predictions)
    probabilities_array = np.array(probabilities)
    
    # Majority voting for classification
    ensemble_predictions = np.round(np.mean(predictions_array, axis=0))
    
    # Average probabilities
    ensemble_probabilities = np.mean(probabilities_array, axis=0)
    
    # Evaluate ensemble
    results = evaluate_model(y_test, ensemble_predictions, ensemble_probabilities, 
                            f"Random Forest Ensemble ({n_models} models)")
    
    return models[-1], ensemble_predictions, ensemble_probabilities, results

# Cross-validation for robust evaluation
def cross_validate_random_forest(X, y, n_splits=5):
    """
    Perform cross-validation for Random Forest
    """
    print(f"\n{'='*60}")
    print(f"{n_splits}-FOLD CROSS VALIDATION")
    print(f"{'='*60}")
    
    # Initialize model
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    
    # Stratified K-Fold for imbalanced data
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    cv_scores = {
        'accuracy': [],
        'precision': [],
        'recall': [],
        'f1': [],
        'roc_auc': []
    }
    
    fold = 1
    for train_idx, val_idx in skf.split(X, y):
        print(f"\nFold {fold}/{n_splits}:")
        
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]
        
        # Train on fold
        rf.fit(X_train_fold, y_train_fold)
        
        # Predict on validation fold
        y_pred = rf.predict(X_val_fold)
        y_scores = rf.predict_proba(X_val_fold)[:, 1]
        
        # Calculate metrics
        cv_scores['accuracy'].append(accuracy_score(y_val_fold, y_pred))
        cv_scores['precision'].append(precision_score(y_val_fold, y_pred, zero_division=0))
        cv_scores['recall'].append(recall_score(y_val_fold, y_pred, zero_division=0))
        cv_scores['f1'].append(f1_score(y_val_fold, y_pred, zero_division=0))
        cv_scores['roc_auc'].append(roc_auc_score(y_val_fold, y_scores))
        
        fold += 1
    
    # Print results
    print(f"\nCross-Validation Results ({n_splits}-fold):")
    for metric, scores in cv_scores.items():
        print(f"{metric.capitalize()}:")
        print(f"  Scores: {[f'{s:.4f}' for s in scores]}")
        print(f"  Mean: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
        print()
    
    return cv_scores

# Main training pipeline
def main_random_forest_pipeline():
    """
    Complete Random Forest training pipeline
    """
    
    print("Starting Random Forest Training Pipeline...")
    print(f"{'='*60}")
    print(f"Training data shape: {X_train.shape}")
    print(f"Test data shape: {X_test.shape}")
    # print(f"Features: {len(FINAL_FEATURES)}")
    print(f"Training class distribution: {np.bincount(y_train)}")
    print(f"Test class distribution: {np.bincount(y_test)}")
    
    # 1. Train basic Random Forest models
    print("\n1. Training basic Random Forest models...")
    rf_models, rf_results = train_random_forests(X_train, y_train, X_test, y_test)
    
    # # 2. Hyperparameter tuning (optional)
    # print("\n2. Performing hyperparameter tuning...")
    # best_rf, tuning_results, search_obj = tune_random_forest(
    #     X_train, y_train, X_test, y_test, 
    #     method='random', cv_folds=3
    # )
    best_rf = rf_models['rf_tuned']
    tuning_results = rf_results['rf_tuned']
    
    # 3. Cross-validation
    print("\n3. Performing cross-validation...")
    cv_results = cross_validate_random_forest(X_train, y_train, n_splits=5)
    
    # 4. Feature importance analysis
    print("\n4. Analyzing feature importance...")
    # Use the best model for feature importance
    best_model_name = max(rf_results, key=lambda x: rf_results[x].get('roc_auc', 0))
    best_model = rf_models[best_model_name]
    
    feature_importance_df, idx_80, idx_90 = analyze_feature_importance(
        best_model, FINAL_FEATURES, top_n=10
    )
    
    # 5. Create ensemble (optional)
    print("\n5. Creating ensemble of Random Forests...")
    ensemble_models, _, _, ensemble_results = create_ensemble_rf(
        X_train, y_train, X_test, y_test, n_models=3
    )
    
    # 6. Compare all models
    print(f"\n{'='*60}")
    print("MODEL COMPARISON SUMMARY")
    print(f"{'='*60}")
    
    comparison_df = pd.DataFrame({
        'Model': list(rf_results.keys()) + ['rf_tuned', 'ensemble'],
        'Accuracy': [r.get('accuracy', 0) for r in rf_results.values()] + 
                    [tuning_results.get('accuracy', 0), ensemble_results.get('accuracy', 0)],
        'Precision': [r.get('precision', 0) for r in rf_results.values()] + 
                     [tuning_results.get('precision', 0), ensemble_results.get('precision', 0)],
        'Recall': [r.get('recall', 0) for r in rf_results.values()] + 
                  [tuning_results.get('recall', 0), ensemble_results.get('recall', 0)],
        'F1-Score': [r.get('f1', 0) for r in rf_results.values()] + 
                    [tuning_results.get('f1', 0), ensemble_results.get('f1', 0)],
        'ROC-AUC': [r.get('roc_auc', 0) for r in rf_results.values()] + 
                   [tuning_results.get('roc_auc', 0), ensemble_results.get('roc_auc', 0)]
    }).sort_values('ROC-AUC', ascending=False)
    
    print(comparison_df.to_string(index=False))
    
    # Visualization
    plot_model_comparison(comparison_df)
    
    return {
        'models': rf_models,
        'best_model': best_model,
        'feature_importance': feature_importance_df,
        'comparison': comparison_df,
        'cv_results': cv_results,
        'ensemble_models': ensemble_models
    }

def plot_model_comparison(comparison_df):
    """
    Plot comparison of different models
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors = plt.cm.Set3(np.linspace(0, 1, len(comparison_df)))
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx // 2, idx % 2]
        bars = ax.barh(comparison_df['Model'], comparison_df[metric], color=colors)
        ax.set_xlabel(metric)
        ax.set_title(f'{metric} Comparison')
        ax.grid(True, alpha=0.3, axis='x')
        
        # Add value labels
        for bar in bars:
            width = bar.get_width()
            ax.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                   f'{width:.3f}', ha='left', va='center')
    
    plt.tight_layout()
    plt.show()

# Utility functions
def save_random_forest_model(model, model_name, path="/kaggle/working/models/"):
    """Save trained Random Forest model"""
    import os
    os.makedirs(path, exist_ok=True)
    
    filename = f"{path}{model_name}.pkl"
    joblib.dump(model, filename)
    print(f"Model saved to {filename}")

def load_random_forest_model(filename):
    """Load saved Random Forest model"""
    return joblib.load(filename)

def predict_with_confidence(model, X_new):
    """
    Make predictions with confidence scores
    """
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(X_new)
        predictions = model.predict(X_new)
        
        # Get confidence (max probability)
        confidence = np.max(probabilities, axis=1)
        
        return predictions, probabilities, confidence
    else:
        predictions = model.predict(X_new)
        return predictions, None, None
# Run complete Random Forest pipeline
results = main_random_forest_pipeline()

# Save the best model
best_model = results['best_model']
save_random_forest_model(best_model, "best_random_forest")
# Save the best model
e_model = results['ensemble_models']
save_random_forest_model(e_model, "ensemble_random_forest")
# Make predictions on test data
predictions, probabilities, confidence = predict_with_confidence(best_model, X_test)

print(f"\nPredictions on test set:")
print(f"Shape: {predictions.shape}")
print(f"Sample predictions: {predictions[:10]}")
print(f"Sample confidence scores: {confidence[:10] if confidence is not None else 'N/A'}")
